# C2_language_models_synthetic_signals: Language models and synthetic signal detection

Public appendix notebook from the Bachelor's Thesis *Evaluación robusta de estrategias de inversión: backtesting, sobreajuste y validación estadística*.

This notebook is included for transparency/reproducibility. It was originally developed in Google Colab; paths may need to be adapted for local execution.


In [ ]:
# -*- coding: utf-8 -*-
"""Project Grok.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1urix7sOkIHFrV28nCy8N7FefKo8peMl6
"""

!pip -q install openai pandas matplotlib

import os
import json
import time
import getpass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from openai import OpenAI
print("Imports ok")

def simulate_gbm(S0=100, mu=0.0, sigma=0.2, T=1.0, N=252, seed=None):
    if seed is not None:
        np.random.seed(seed)

    dt = T / N
    t = np.linspace(0, T, N + 1)

    S = np.zeros(N + 1)
    S[0] = S0

    for i in range(1, N + 1):
        Z = np.random.normal(0, 1)
        S[i] = S[i-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)

    return t, S
t, S = simulate_gbm(mu=0.1, sigma=0.2, seed=42)
print(len(S), S[:5])

plt.plot(t, S)
plt.title("Ejemplo trayectoria GBM")
plt.grid(True)
plt.show()

def generate_scenarios(n_per_type=10, sigma=0.2, T=1.0, N=252):
    scenarios = []

    types = [
        {"type": "no_signal", "mu": 0.0},
        {"type": "positive_drift", "mu": 0.10},
        {"type": "negative_drift", "mu": -0.10},
    ]

    seed = 1200

    for tinfo in types:
        for _ in range(n_per_type):
            _, S = simulate_gbm(
                S0=100,
                mu=tinfo["mu"],
                sigma=sigma,
                T=T,
                N=N,
                seed=seed
            )

            scenarios.append({
                "id": seed,
                "type": tinfo["type"],
                "mu": tinfo["mu"],
                "sigma": sigma,
                "prices": [round(x, 4) for x in S.tolist()]
            })
            seed += 1

    return scenarios
scenarios = generate_scenarios(n_per_type=33)
print("Numero de escenarios:", len(scenarios))
print(scenarios[0]["type"], scenarios[0]["mu"], scenarios[0]["sigma"])
print(scenarios[0]["prices"][:5])

def build_prompt_from_scenario(scenario):
    prices = scenario["prices"]

    prompt = f"""
You are given a financial price series.

Your task is:
1. Decide whether there is an exploitable directional pattern in the series.
2. If there is a pattern, choose one action: "long" or "short".
3. If there is no reliable pattern, choose "flat".
4. Give a short justification.
5. Return only valid JSON.

The JSON must have exactly this structure:
{{
  "has_pattern": true or false,
  "action": "long" or "short" or "flat",
  "confidence": a number between 0 and 1,
  "reasoning": "short explanation"
}}

Price series:
{prices}
"""
    return prompt.strip()
print(build_prompt_from_scenario(scenarios[0])[:1500])

def get_expected_answer(scenario):
    if scenario["type"] == "no_signal":
        return {"has_pattern": False, "action": "flat"}
    elif scenario["type"] == "positive_drift":
        return {"has_pattern": True, "action": "long"}
    elif scenario["type"] == "negative_drift":
        return {"has_pattern": True, "action": "short"}
    else:
        raise ValueError(f"Unknown scenario type: {scenario['type']}")

def score_model_answer(scenario, model_answer):
    expected = get_expected_answer(scenario)

    pattern_correct = model_answer["has_pattern"] == expected["has_pattern"]
    action_correct = model_answer["action"] == expected["action"]
    false_signal = scenario["type"] == "no_signal" and model_answer["has_pattern"] is True

    return {
        "expected_has_pattern": expected["has_pattern"],
        "expected_action": expected["action"],
        "pattern_correct": pattern_correct,
        "action_correct": action_correct,
        "false_signal": false_signal
    }
test_answer = {
    "has_pattern": True,
    "action": "short",
    "confidence": 0.58,
    "reasoning": "Clear downward trend."
}

print(score_model_answer(scenarios[0], test_answer))

def parse_model_response(response_text):
    try:
        text = str(response_text).strip()
        if text.startswith("```json"):
            text = text[len("```json"):].strip()
        elif text.startswith("```"):
            text = text[len("```"):].strip()
        if text.endswith("```"):
            text = text[:-3].strip()

        return json.loads(text)
    except Exception:
        return None

def compute_metrics(scenarios, model_answers):
    valid_pairs = [(s, a) for s, a in zip(scenarios, model_answers) if a is not None]

    results = []
    for scenario, answer in valid_pairs:
        results.append(score_model_answer(scenario, answer))

    pattern_accuracy = float(np.mean([r["pattern_correct"] for r in results])) if results else np.nan
    action_accuracy = float(np.mean([r["action_correct"] for r in results])) if results else np.nan

    no_signal_cases = [
        r for (s, _), r in zip(valid_pairs, results)
        if s["type"] == "no_signal"
    ]
    false_signal_rate = float(np.mean([r["false_signal"] for r in no_signal_cases])) if no_signal_cases else np.nan

    return {
        "pattern_accuracy": pattern_accuracy,
        "action_accuracy": action_accuracy,
        "false_signal_rate": false_signal_rate,
        "n_valid_answers": len(valid_pairs)
    }

def build_results_table(scenarios, model_answers):
    rows = []

    for scenario, answer in zip(scenarios, model_answers):
        if answer is None:
            rows.append({
                "id": scenario["id"],
                "type": scenario["type"],
                "expected_action": get_expected_answer(scenario)["action"],
                "predicted_action": None,
                "expected_pattern": get_expected_answer(scenario)["has_pattern"],
                "predicted_pattern": None,
                "pattern_correct": None,
                "action_correct": None,
                "false_signal": None
            })
            continue

        expected = get_expected_answer(scenario)
        score = score_model_answer(scenario, answer)

        rows.append({
            "id": scenario["id"],
            "type": scenario["type"],
            "expected_action": expected["action"],
            "predicted_action": answer["action"],
            "expected_pattern": expected["has_pattern"],
            "predicted_pattern": answer["has_pattern"],
            "pattern_correct": score["pattern_correct"],
            "action_correct": score["action_correct"],
            "false_signal": score["false_signal"]
        })

    return pd.DataFrame(rows)

dummy_answers = []
for scenario in scenarios:
    expected = get_expected_answer(scenario)
    dummy_answers.append({
        "has_pattern": expected["has_pattern"],
        "action": expected["action"],
        "confidence": 0.7,
        "reasoning": "Dummy perfect answer"
    })

print(compute_metrics(scenarios, dummy_answers))
build_results_table(scenarios, dummy_answers).head()

GROQ_API_KEY = getpass.getpass("Pega tu Groq API key: ")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)
print("Cliente Groq creado")

resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "Return only valid JSON."},
        {"role": "user", "content": 'Return exactly this JSON: {"ok": true}'}
    ],
    temperature=0
)

print(resp.choices[0].message.content)

def query_llm_groq(prompt, model="openai/gpt-oss-120b", sleep_seconds=0.5):
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You must return only valid JSON and no extra text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    time.sleep(sleep_seconds)
    return resp.choices[0].message.content


# --- EJECUCIÓN ---
try:
    test_output = query_llm_groq(build_prompt_from_scenario(scenarios[0]))
    print("RAW OUTPUT:\n", test_output)

    parsed_output = parse_model_response(test_output)
    print("\nPARSED OUTPUT:\n", parsed_output)

except Exception as e:
    print("ERROR REAL:", e)

sample_scenarios = scenarios[:150]

raw_outputs = []
parsed_answers = []

for i, scenario in enumerate(sample_scenarios):
    prompt = build_prompt_from_scenario(scenario)
    output = query_llm_groq(prompt)

    raw_outputs.append(output)
    parsed_answers.append(parse_model_response(output))

    print(f"Escenario {i} ({scenario['type']}) listo")
raw_outputs[:2]

raw_outputs.append(output)
    parsed_answers.append(parse_model_response(output))

    print(f"Escenario {i} ({scenario['type']}) listo")
raw_outputs[:2]

# Celda 12 — resumen robusto usando raw_outputs (NO parsed_answers)

import json
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Helpers ----------
def clean_json_response(text):
    if text is None:
        return None
    text = str(text).strip()
    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    elif text.startswith("```"):
        text = text[len("```"):].strip()
    if text.endswith("```"):
        text = text[:-3].strip()
    return text

def safe_parse_answer(text):
    try:
        cleaned = clean_json_response(text)
        return json.loads(cleaned)
    except Exception:
        return {
            "has_pattern": None,
            "action": None,
            "confidence": None,
            "reasoning": f"Parse error: {str(text)[:120]}"
        }

def expected_action_from_type(scenario_type):
    if scenario_type == "no_signal":
        return "flat"
    elif scenario_type == "positive_drift":
        return "long"
    elif scenario_type == "negative_drift":
        return "short"
    return None

def expected_has_pattern_from_type(scenario_type):
    if scenario_type == "no_signal":
        return False
    elif scenario_type in ["positive_drift", "negative_drift"]:
        return True
    return None

# ---------- IMPORTANTE ----------
# Usamos raw_outputs porque parsed_answers contiene muchos None
n_scenarios = len(sample_scenarios)
n_raw_answers = len(raw_outputs)
n_used = min(n_scenarios, n_raw_answers)

used_scenarios = sample_scenarios[:n_used]
used_answers = [safe_parse_answer(x) for x in raw_outputs[:n_used]]

# ---------- Construcción de tabla ----------
rows = []
for sc, ans in zip(used_scenarios, used_answers):
    expected_action = expected_action_from_type(sc["type"])
    expected_has_pattern = expected_has_pattern_from_type(sc["type"])

    pred_action = ans.get("action")
    pred_has_pattern = ans.get("has_pattern")
    pred_confidence = ans.get("confidence")
    pred_reasoning = ans.get("reasoning")

    pattern_correct = None if pred_has_pattern is None else (pred_has_pattern == expected_has_pattern)
    action_correct = None if pred_action is None else (pred_action == expected_action)

    rows.append({
        "id": sc["id"],
        "type": sc["type"],
        "mu": sc["mu"],
        "sigma": sc["sigma"],
        "expected_has_pattern": expected_has_pattern,
        "predicted_has_pattern": pred_has_pattern,
        "pattern_correct": pattern_correct,
        "expected_action": expected_action,
        "predicted_action": pred_action,
        "action_correct": action_correct,
        "confidence": pred_confidence,
        "reasoning": pred_reasoning
    })

df_sample = pd.DataFrame(rows)

# ---------- Métricas globales ----------
valid_pattern = df_sample["pattern_correct"].dropna()
valid_action = df_sample["action_correct"].dropna()

pattern_accuracy = valid_pattern.mean() if len(valid_pattern) > 0 else float("nan")
action_accuracy = valid_action.mean() if len(valid_action) > 0 else float("nan")

print("=" * 80)
print("GLOBAL RESULTS")
print("=" * 80)
print(f"Requested scenarios    : {n_scenarios}")
print(f"Raw answers received   : {n_raw_answers}")
print(f"Matched pairs used     : {n_used}")
print(f"Pattern accuracy       : {pattern_accuracy:.2%}" if pd.notna(pattern_accuracy) else "Pattern accuracy       : N/A")
print(f"Action accuracy        : {action_accuracy:.2%}" if pd.notna(action_accuracy) else "Action accuracy        : N/A")
print()

# ---------- Resumen por tipo ----------
summary_by_type = (
    df_sample
    .groupby("type")
    .agg(
        n_cases=("id", "count"),
        pattern_accuracy=("pattern_correct", "mean"),
        action_accuracy=("action_correct", "mean")
    )
    .reset_index()
)

print("=" * 80)
print("RESULTS BY SCENARIO TYPE")
print("=" * 80)
display(summary_by_type.style.format({
    "pattern_accuracy": "{:.2%}",
    "action_accuracy": "{:.2%}"
}))

print("=" * 80)
print("ACTION COUNTS BY TYPE")
print("=" * 80)
action_counts = (
    df_sample
    .groupby(["type", "predicted_action"])
    .size()
    .unstack(fill_value=0)
)
display(action_counts)

print("=" * 80)
print("DETAILED PREDICTIONS")
print("=" * 80)
display(df_sample)

# ---------- Gráficos ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_names = ["Pattern Accuracy", "Action Accuracy"]
metric_values = [
    pattern_accuracy if pd.notna(pattern_accuracy) else 0,
    action_accuracy if pd.notna(action_accuracy) else 0
]

axes[0].bar(metric_names, metric_values)
axes[0].set_ylim(0, 1)
axes[0].set_title("Global Metrics")
axes[0].grid(axis="y", alpha=0.3)

for i, v in enumerate(metric_values):
    axes[0].text(i, min(v + 0.02, 0.98), f"{v:.2%}", ha="center")

axes[1].bar(summary_by_type["type"], summary_by_type["action_accuracy"].fillna(0))
axes[1].set_ylim(0, 1)
axes[1].set_title("Action Accuracy by Scenario Type")
axes[1].grid(axis="y", alpha=0.3)

for i, v in enumerate(summary_by_type["action_accuracy"].fillna(0)):
    axes[1].text(i, min(v + 0.02, 0.98), f"{v:.2%}", ha="center")

plt.tight_layout()
plt.show()

with open("groq_raw_outputs.json", "w") as f:
    json.dump(raw_outputs, f, indent=2)

with open("groq_parsed_answers.json", "w") as f:
    json.dump(parsed_answers, f)

print("Guardado")

!ls

df_sample["predicted_action"].value_counts()

pd.crosstab(df_sample["type"], df_sample["predicted_action"], normalize="index")
